## Standard Augmentation for EEG Data

In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path 
import mne

from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [41]:
def find_project_root():
    p = Path.cwd()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    raise FileNotFoundError("Project root (with .git) not found")

PROJECT_ROOT = find_project_root()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "nm000114"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed_features"
RESULTS_DIR = PROJECT_ROOT / "results"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

PROJECT_ROOT: /home/srlee185/MSSE-277b-final-project-
RAW_DATA_DIR: /home/srlee185/MSSE-277b-final-project-/data/raw/nm000114
PROCESSED_DIR: /home/srlee185/MSSE-277b-final-project-/data/processed_features
RESULTS_DIR: /home/srlee185/MSSE-277b-final-project-/results


In [42]:
# Helper function to load a single EDF file and return the raw data object

def get_eeg_file(subject_id: str, condition: str):
    return RAW_DATA_DIR / subject_id / "eeg" / f"{subject_id}_task-{condition}_eeg.edf"

In [43]:
# Scan all files in the raw data directory to ensure we can access them

edf_files = sorted(RAW_DATA_DIR.glob("sub-*/*/*.edf"))

In [44]:
# Parser cell to read all EDF files and extract data and labels for modeling

def infer_label_from_subject(subject_id: str) -> int:
    """
    Infer the label (0 or 1) from the subject ID.
    returns: 
        0 for healthy controls (sub-HS*)
        1 for MDD patients (sub-MDD*)
    """
    subject_upper = subject_id.upper()
    if "HS" in subject_upper:
        return 0
    elif "MDD" in subject_upper:
        return 1 
    else: 
        raise ValueError(f"Could not infer label from subject ID: {subject_id}")
    
def parse_condition_from_filename(filepath: Path) -> str:
    """
    Parse the condition from the filename
    Returns:
        - eyesClosed
        - eyesOpen
        - P300
    """

    name = filepath.name 
    if "task-eyesClosed" in name:
        return "eyesClosed"
    elif "task-eyesOpen" in name:
        return "eyesOpen"
    elif "task-P300" in name:
        return "P300"
    else:
        raise ValueError(f"Could not parse condition from filename: {name}")

In [45]:
# metadata list to hold all parsed information 

rows = []

for filepath in edf_files:
    subject_id = filepath.parts[-3]  # e.g. subHS1
    condition = parse_condition_from_filename(filepath)
    label = infer_label_from_subject(subject_id)

    rows.append({
        "patient_id": subject_id,
        "recording_id": filepath.stem,
        "label": label,
        "condition": condition,
        "filepath": str(filepath),
    })


# Create a DataFrame from the metadata
metadata_df = pd.DataFrame(rows)
print(metadata_df.head())

  patient_id                  recording_id  label   condition  \
0    sub-HS1         sub-HS1_task-P300_eeg      0        P300   
1    sub-HS1   sub-HS1_task-eyesClosed_eeg      0  eyesClosed   
2    sub-HS1     sub-HS1_task-eyesOpen_eeg      0    eyesOpen   
3   sub-HS10        sub-HS10_task-P300_eeg      0        P300   
4   sub-HS10  sub-HS10_task-eyesClosed_eeg      0  eyesClosed   

                                            filepath  
0  /home/srlee185/MSSE-277b-final-project-/data/r...  
1  /home/srlee185/MSSE-277b-final-project-/data/r...  
2  /home/srlee185/MSSE-277b-final-project-/data/r...  
3  /home/srlee185/MSSE-277b-final-project-/data/r...  
4  /home/srlee185/MSSE-277b-final-project-/data/r...  


In [46]:
COMMON_CHANNELS = [
'EEG Fp1-LE', 'EEG F3-LE', 'EEG C3-LE', 'EEG P3-LE', 'EEG O1-LE',
 'EEG F7-LE', 'EEG T3-LE', 'EEG T5-LE', 'EEG Fz-LE', 'EEG Fp2-LE', 
 'EEG F4-LE', 'EEG C4-LE', 'EEG P4-LE', 'EEG O2-LE', 'EEG F8-LE', 
 'EEG T4-LE', 'EEG T6-LE', 'EEG Cz-LE', 'EEG Pz-LE', 'EEG A2-A1'
]

print("Number of common channels:", len(COMMON_CHANNELS))

Number of common channels: 20


In [47]:
# Step 1: Extract spectral features (e.g. power in different frequency bands) from each EEG recording

def extract_band_power(raw, COMMON_CHANNELS) -> np.ndarray:
    """
    Extract basic EEG band power features from the raw MNE objects.
    Returns a feature vector.
    """

    data_raw = raw.copy()
    data_raw.pick(COMMON_CHANNELS)
    
    # get data 
    data = data_raw.get_data()      # shape: (n_channels, n_time points)
    sfreq = data_raw.info['sfreq']  # sampling frequency (e.g. 256 Hz)

    # Define frequency bands 
    bands = {
        "delta": (1, 4),     # deep sleep
        "theta": (4, 8),     # drowsiness
        "alpha": (8, 13),    # relaxed 
        "beta":  (13, 30)    # active thinking 
    }

    all_features = []

    # Loop through each channel (ch) and convert signal to frequency domain using Welch's method to estimate power spectral density (psd)
    for ch in data:        
        psd, freqs = mne.time_frequency.psd_array_welch(
            ch,
            sfreq=sfreq,
            fmin=1,
            fmax=30,
            verbose=False
        )

        total_power = psd.sum()
        
        band_features = [
            (
                psd[(freqs >= fmin) & (freqs <= fmax)].mean()
            / total_power                                      # normalize by total power to get relative power in each band
            if total_power > 0 else 0                          # handle case where total power is zero to avoid division by zero
            )                           
            for (fmin, fmax) in bands.values()
            ]
    
        all_features.append(band_features)

    return np.nan_to_num(np.array(all_features).flatten())


In [278]:
# Build feature matrix 

records = [
    (
        extract_band_power(
            mne.io.read_raw_edf(row["filepath"], preload=True, verbose=False),
            COMMON_CHANNELS
        ),
        row["label"],
        row["patient_id"]
    )
    for _, row in metadata_df.iterrows()
]

X = np.vstack([r[0] for r in records])
y = np.array([r[1] for r in records])
groups = np.array([r[2] for r in records])


In [270]:
# Baseline augmentation
gfk = GroupKFold(n_splits=5)

pipeline_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf",probability=True)
     )
])

# Standard Augmentation

In [48]:
# # Creating a DataFrame to read outputs easier
# record_with_cond = [
#     (
#         extract_band_power(
#             mne.io.read_raw_edf(row["filepath"], preload=True, verbose=False),
#             COMMON_CHANNELS
#         ),
#         row["label"],
#         row["patient_id"],
#         row["condition"]
#     )
#     for _, row in metadata_df.iterrows()
# ]

# records_restructured = []

# for rel_power, label, patient, condition in record_with_cond:
#     rel_power = rel_power.reshape(20,4)
#     records_restructured.append((patient, condition, rel_power, label))

# readable_data = pd.DataFrame(records_restructured, columns=['Patient', 'Condition', 'Relative_Power', 'Label'])
# readable_data['Relative_Power'] = readable_data['Relative_Power'].apply(lambda x: x.tolist())
# readable_data = readable_data.explode('Relative_Power')
# readable_data[['rel_power_delta', 'rel_power_theta', 'rel_power_alpha', 'rel_power_beta']] = pd.DataFrame(readable_data['Relative_Power'].tolist(), index=readable_data.index)
# readable_data = readable_data.drop(columns=['Relative_Power'])
# readable_data['EEG_Channel'] = COMMON_CHANNELS * 181

In [63]:
# Creating time masking function

def time_masking(data, mask_size, noise=False): # Data must be raw edf file data
    n_channels, n_times = data.shape
    # start = np.random.randint(0, n_times-mask_size)

    if noise == False: # Masking with zeroes
        data[:, 0:mask_size] = 0
    else:              # Masking with gaussain noise
        data[:, 0:mask_size] = np.random.normal(
    loc=0, scale=0.1 * data.std(), size=(n_channels, mask_size)
)
    return data

In [50]:
def time_masking_band_power(filepath, COMMON_CHANNELS) -> np.ndarray:
    """
    Extract basic EEG band power features from the raw MNE objects.
    Returns a feature vector.
    """

    raw = mne.io.read_raw_edf(filepath, preload=True, verbose=False)
    data_raw = raw.copy()
    data_raw.pick(COMMON_CHANNELS)
    data = data_raw.get_data()
    sfreq = data_raw.info['sfreq']  # sampling frequency (e.g. 256 Hz)

    time_masked = time_masking(data, 900, False) # 900

    # Define frequency bands 
    bands = {
        "delta": (1, 4),     # deep sleep
        "theta": (4, 8),     # drowsiness
        "alpha": (8, 13),    # relaxed 
        "beta":  (13, 30)    # active thinking 
    }

    all_features = []

    # Loop through each channel (ch) and convert signal to frequency domain using Welch's method to estimate power spectral density (psd)
    for ch in time_masked:        
        psd, freqs = mne.time_frequency.psd_array_welch(
            ch,
            sfreq=sfreq,
            fmin=1,
            fmax=30,
            verbose=False
        )

        total_power = psd.sum()
        
        band_features = [
            (
                psd[(freqs >= fmin) & (freqs <= fmax)].mean()
            / total_power                                      # normalize by total power to get relative power in each band
            if total_power > 0 else 0                          # handle case where total power is zero to avoid division by zero
            )                           
            for (fmin, fmax) in bands.values()
            ]
    
        all_features.append(band_features)

    return np.nan_to_num(np.array(all_features).flatten())


In [51]:
# Applying the time mask augmentation
time_mask_xy = [
    (
        time_masking_band_power(row['filepath'], COMMON_CHANNELS),
        row['label'],
        row['patient_id']
        )
        for _, row in metadata_df.iterrows()
    ]

time_mask_x = np.vstack([r[0] for r in time_mask_xy])
y = np.array([r[1] for r in time_mask_xy])
groups = np.array([r[2] for r in time_mask_xy])


# create the splitter, five folds total 
gfk = GroupKFold(n_splits=5)

pipeline_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf",probability=True)
     )
])

results_svm_time = cross_validate(
    pipeline_svm,
    time_mask_x,
    y,
    cv=gfk.split(time_mask_x, y, groups),
    scoring=["accuracy", "f1", "roc_auc"]
)

print("SVM accuracy:", results_svm_time["test_accuracy"])
print("SVM mean accuracy:", results_svm_time["test_accuracy"].mean())
print("SVM std accuracy:", results_svm_time["test_accuracy"].std())

print("SVM F1 score:", results_svm_time["test_f1"])
print("SVM mean F1 score:", results_svm_time["test_f1"].mean())  

print("SVM ROC-AUC:", results_svm_time["test_roc_auc"])
print("SVM mean ROC-AUC:", results_svm_time["test_roc_auc"].mean())

SVM accuracy: [0.94594595 0.78378378 0.69444444 0.77142857 0.63888889]
SVM mean accuracy: 0.7668983268983269
SVM std accuracy: 0.10392273375781541
SVM F1 score: [0.94736842 0.82608696 0.71794872 0.82608696 0.62857143]
SVM mean F1 score: 0.7892124961232513
SVM ROC-AUC: [0.9619883  0.83540373 0.846875   0.90816327 0.753125  ]
SVM mean ROC-AUC: 0.8611110592215528


In [52]:
# Testing pipeline with only p300 patients

p300_mask = metadata_df['condition'] == "P300"

time_mask_x_p300 = time_mask_x[p300_mask]
y_p300 = y[p300_mask]
groups_p300 = groups[p300_mask]

print("X_p300 shape:", time_mask_x_p300.shape)
print("y_p300 shape:", y_p300.shape)
print("unique patients:", len(np.unique(groups_p300)))

X_p300 shape: (61, 80)
y_p300 shape: (61,)
unique patients: 61


In [53]:
pipeline_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf",probability=True)
     )
])

results_svm_time_p300 = cross_validate(
    pipeline_svm,
    time_mask_x_p300,
    y_p300,
    cv=gfk.split(time_mask_x_p300, y_p300, groups_p300),
    scoring=["accuracy", "f1", "roc_auc"]
)

print("SVM accuracy:", results_svm_time_p300["test_accuracy"])
print("SVM mean accuracy:", results_svm_time_p300["test_accuracy"].mean())
print("SVM std accuracy:", results_svm_time_p300["test_accuracy"].std())

print("SVM F1 score:", results_svm_time_p300["test_f1"])
print("SVM mean F1 score:", results_svm_time_p300["test_f1"].mean())  

print("SVM ROC-AUC:", results_svm_time_p300["test_roc_auc"])
print("SVM mean ROC-AUC:", results_svm_time_p300["test_roc_auc"].mean())

SVM accuracy: [0.76923077 0.91666667 0.66666667 0.83333333 0.83333333]
SVM mean accuracy: 0.8038461538461539
SVM std accuracy: 0.08304684482411102
SVM F1 score: [0.8        0.93333333 0.75       0.85714286 0.85714286]
SVM mean F1 score: 0.8395238095238096
SVM ROC-AUC: [0.92857143 1.         0.88888889 0.88888889 0.82857143]
SVM mean ROC-AUC: 0.906984126984127


## Discussion

There is a noticeable improvement in the model when masking the first 900 timepoints. However, these improvements roughly are only +0.1 - 0.2 % in accuracy, are negligible changes. There is a meaningful increase in ROC-AUC score, indicating the model has been improved in distinguishing between classes (ROC-AUC increased by ~6%). There is space investigating thes augmenations further, especially combing multiple together.

In [245]:
# Functions for Amplitude Scaling and Gaussain Noise

def gaussian_noise(data, noise_level=0.01):
    noise = np.random.normal(0, noise_level * data.std(), data.shape)
    return data + noise


def amplitude_scaling(data, scale_range=(0,2)):
    scale = np.random.uniform(*scale_range)
    return data * scale

In [293]:
# Applying Amplitude Scaling and Gaussian Noise with Time Masking

amplitude_X = amplitude_scaling(X)
gaussian_X = gaussian_noise(X)

gaussian_time_X = gaussian_noise(time_mask_x)
amp_time_X = amplitude_scaling(time_mask_x)
amp_gaussian_X = gaussian_noise(amplitude_scaling(X))
gaussian_amp_X = amplitude_scaling(gaussian_noise(X))

gaussian_time_X_p300 = gaussian_time_X[p300_mask]
amp_time_X_p300 = amp_time_X[p300_mask]
amp_gaussian_X_p300 = amp_gaussian_X[p300_mask]
gaussian_amp_X_p300 =gaussian_amp_X[p300_mask]
amp_X_p300 = amplitude_X[p300_mask]
gau_X_p300 = gaussian_X[p300_mask]

def run_augmented_svm(condition: str, X, y = y, groups = groups, pipeline = pipeline_svm):
    results = cross_validate(
        pipeline,
        X,
        y,
        cv=gfk.split(X, y, groups),
        scoring=["accuracy", "f1", "roc_auc"]
    )
    return [condition, results["test_accuracy"].mean(), results["test_f1"].mean(), results["test_roc_auc"].mean()]

baseline = run_augmented_svm("Baseline", X)
time_mask = run_augmented_svm("Time Mask", time_mask_x)
amp = run_augmented_svm("Amplitude Scaling", amplitude_X)
gau = run_augmented_svm("Gaussian Noise", gaussian_X)


gaussian_time = run_augmented_svm("Gaussain+Time", gaussian_time_X)
amp_time = run_augmented_svm("Amplitude+Time", amp_time_X)
amp_gaussian = run_augmented_svm("Amplitude then Gaussian", amp_gaussian_X)
gaussain_amp = run_augmented_svm("Gaussian then Amplitude", gaussian_amp_X)

time_mask_p300 = run_augmented_svm("P300 Time", time_mask_x_p300, y_p300, groups_p300)
amp_p300 = run_augmented_svm("P300 Amplitude Scaling", amp_gaussian_X_p300, y_p300, groups_p300)
gau_p300 = run_augmented_svm("P300 Gaussain Noise", gau_X_p300, y_p300, groups_p300)
gaussian_time_p300 = run_augmented_svm("P300 Gaussian+Time", gaussian_amp_X_p300, y_p300, groups_p300)
amp_time_p300 = run_augmented_svm("P300 Amplitude+time", amp_gaussian_X_p300, y_p300, groups_p300)
amp_gaussian_p300 = run_augmented_svm("P300 Amplitude then Gaussain", amp_gaussian_X_p300, y_p300, groups_p300)
gaussain_amp_p300 = run_augmented_svm("P300 Gaussain then Amplitude", gaussian_amp_X_p300, y_p300, groups_p300)

In [295]:
results_df = pd.DataFrame([
    baseline, 
    time_mask, amp, gau, gaussian_time, amp_time, amp_gaussian, gaussain_amp,
    time_mask_p300, amp_p300, gau_p300, gaussian_time_p300, amp_time_p300, amp_gaussian_p300, gaussain_amp_p300, 
], columns=["Condition", "Accuracy", "F1", "ROC-AUC"])

results_df

,Condition,Accuracy,F1,ROC-AUC
0,Baseline,0.739271,0.772340,0.850540
1,Time Mask,0.766898,0.789212,0.861111
2,Amplitude Scaling,0.739271,0.772340,0.850540
3,Gaussian Noise,0.739271,0.772340,0.856798
4,Gaussain+Time,0.766748,0.789294,0.861060
5,Amplitude+Time,0.766898,0.789212,0.861111
6,Amplitude then Gaussian,0.733865,0.770583,0.849652
7,Gaussian then Amplitude,0.728310,0.764522,0.851041
8,P300 Time,0.803846,0.839524,0.906984
9,P300 Amplitude Scaling,0.803846,0.839524,0.900476
